[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lugerlakes/signal_analysis_course/blob/main/Lab1_Datos_PDW_pandas.ipynb)

<div style="border-left:6px solid #1F4E79;padding:10px 16px;background:#F3F7FB">

<span style="font-size:12px;letter-spacing:1px;color:#1F4E79">CURSO DE GUERRA ELECTRÓNICA · ASIGNATURA: ANÁLISIS DE SEÑALES DE RADAR</span>

# Laboratorio 1 · Datos PDW con pandas
### Cargar, explorar y limpiar una captura de un receptor ESM

<span style="font-size:12px;color:#7a7a7a">MATERIAL DE INSTRUCCIÓN · DATOS SINTÉTICOS · Emisores y bibliotecas ficticios</span>
</div>

| | |
|---|---|
| ⏱️ **Duración estimada** | 2 a 3 horas |
| 📥 **Datos de entrada** | `D1_ESM_captura_emisor_unico.xlsx` |
| 📤 **Producto al final** | PDW limpio + checklist de preprocesamiento + primeras observaciones del emisor |

**Objetivos de aprendizaje** — al terminar este laboratorio serás capaz de:

1. Explicar qué es un **PDW** y qué mide un receptor ESM/ELINT.
2. Cargar un archivo Excel exportado por un sistema ESM en un **DataFrame** de pandas.
3. Explorar la tabla: dimensiones, columnas, tipos, primeras y últimas filas.
4. Detectar y corregir los problemas típicos de una exportación real: nombres corruptos, columnas vacías, celdas vacías, registros "fantasma", duplicados y desorden.
5. Calcular parámetros derivados (**TOA en segundos, DTOA**) y hacer los primeros gráficos temporales.
6. Guardar el PDW limpio para las etapas siguientes.

**Cómo leer este cuaderno**

| Ícono | Significado |
|---|---|
| 📘 | Concepto o teoría breve |
| ▶️ | Celda de código para **ejecutar** (clic en la celda y `Shift + Enter`) |
| ✏️ | **Tu turno**: completa el código o responde |
| 💡 | Consejo práctico |
| ⚠️ | Error frecuente o precaución |
| 🎯 | Conexión con la Guerra Electrónica (ESM / ELINT) |

> Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla de forma extraña: menú **Entorno de ejecución → Reiniciar y ejecutar todo**.

---
## 1. Contexto: ¿qué entrega un receptor ESM?

📘 Un receptor **ESM** (*Electronic Support Measures*) o **ELINT** (*Electronic Intelligence*) no "ve" la imagen del radar: mide **cada pulso** que llega a su antena y lo resume en un registro llamado **PDW** (*Pulse Descriptor Word*).

| Campo del PDW | Nombre | Unidad típica | ¿Qué nos dice? |
|---|---|---|---|
| **TOA** | *Time Of Arrival* – instante de llegada | µs | Con TOA consecutivos se calcula la **PRI** |
| **RF** | Frecuencia portadora | MHz | Banda, agilidad en frecuencia |
| **PW** | *Pulse Width* – ancho de pulso | µs | Resolución, modo de operación |
| **PA / AMP** | Amplitud (potencia recibida) | dBm | Patrón de antena → **tipo de exploración (scan)** |
| **AOA / DOA** | Ángulo (dirección) de llegada | grados | Ubicación angular del emisor |

🎯 **ESM vs ELINT.** Ambos miden lo mismo. El **ESM** prioriza la reacción en tiempo real (alerta, autoprotección); el **ELINT** hace análisis detallado *a posteriori* para caracterizar emisores y **alimentar las bibliotecas** que usa el ESM.

### 1.1 Dónde estamos en el proceso de análisis

| Fase | Contenido | Laboratorios |
|---|---|---|
| Colección | El sensor intercepta y registra los PDW | (datos entregados) |
| **Producción** · 1. Preprocesamiento | Carga, limpieza, unidades, orden | **Lab 1** ← *estamos aquí* |
| **Producción** · 2. Extracción de características | Estadística, PRI, scan, RF, PW | Labs 2 y 3 |
| **Producción** · 3. Identificación | Separación de emisores, conocidos y desconocidos | Labs 4 y 5 |
| **Producción** · 4. Reporte preliminar | Informe de interceptación | Lab 6 |
| Difusión | Entrega del informe a quien corresponda | — |

---
## 2. Obtener el archivo de datos

**En Google Colab** el archivo se **descarga solo** desde el repositorio del curso (celda siguiente). Si no hay internet o prefieres tus propios archivos:

1. **Subir el archivo**: clic en el ícono de carpeta 📁 (barra izquierda) → botón de *subir* → elige `D1_ESM_captura_emisor_unico.xlsx`. *(Los archivos subidos se borran al cerrar la sesión.)*
2. **Google Drive**: copia la carpeta `datos` a tu Drive y ejecuta `from google.colab import drive; drive.mount('/content/drive')`.

**En Jupyter local**: deja la carpeta `datos/` junto a este cuaderno.

▶️ La siguiente celda busca el archivo en los lugares habituales; si no lo encuentra, lo **descarga del repositorio del curso** y, si tampoco puede (sin internet), te pide subirlo. *(Fíjate: usa lo que aprendiste en el Lab 0: lista, `for`, `if`.)*

In [ ]:
import os

ARCHIVO = "D1_ESM_captura_emisor_unico.xlsx"
URL_DATOS = "https://raw.githubusercontent.com/lugerlakes/signal_analysis_course/main/datos/"
carpetas = [".", "datos", "/content", "/content/datos", "/content/sample_data",
            "/content/drive/MyDrive", "/content/drive/MyDrive/datos"]

ruta = None
for carpeta in carpetas:
    candidata = os.path.join(carpeta, ARCHIVO)
    if os.path.exists(candidata):
        ruta = candidata
        break

if ruta is None and URL_DATOS:            # 2º intento: descargar desde el repositorio del curso
    import urllib.request
    os.makedirs("datos", exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_DATOS + ARCHIVO, os.path.join("datos", ARCHIVO))
        ruta = os.path.join("datos", ARCHIVO)
        print("⬇️ Archivo descargado desde el repositorio del curso")
    except Exception as e:
        print("⚠️ No se pudo descargar:", e)

if ruta is None:                          # 3º intento: subirlo a mano
    try:
        from google.colab import files
        print("📁 Selecciona el archivo en tu computador...")
        files.upload()
        ruta = ARCHIVO
    except ImportError:
        print("⚠️ No encontré el archivo. Cópialo en una carpeta 'datos' junto al cuaderno.")

print("Archivo a utilizar:", ruta)

---
## 3. pandas y el DataFrame

📘 **pandas** es la librería estándar para trabajar con **tablas**. Su objeto principal es el **DataFrame**: piensa en una hoja de Excel donde

* cada **fila** es un **pulso** interceptado,
* cada **columna** es un **parámetro** del PDW.

Se importa por convención como `pd`. Leer un Excel es una sola línea: `pd.read_excel(ruta)`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

crudo = pd.read_excel(ruta)      # "crudo" = tal como lo exportó el sistema, sin tocar
print("Archivo cargado ✔")

### 3.1 Primer vistazo

| Instrucción | Muestra |
|---|---|
| `df.shape` | (nº filas, nº columnas) |
| `df.head(n)` / `df.tail(n)` | primeras / últimas *n* filas |
| `df.columns` | nombres de las columnas |
| `df.info()` | tipos de dato y celdas no vacías por columna |

In [ ]:
crudo.shape

In [ ]:
crudo.head(10)

In [ ]:
crudo.tail()

In [ ]:
crudo.columns

In [ ]:
crudo.info()

### ✏️ Tu turno 1 · Inspección visual
Mirando las salidas anteriores, anota **al menos 4 problemas** que ves en esta exportación antes de seguir leyendo.

*Tus observaciones (doble clic para editar):*
1. …
2. …
3. …
4. …

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


1. Los nombres de columna tienen caracteres extraños: `TOA(¦Ìs)` en vez de `TOA(µs)`.
2. Hay una columna `Unnamed: 6` casi vacía (con algunos `...`).
3. `PW(¦Ìs)` tiene menos valores no nulos que el resto → **celdas vacías**.
4. Las unidades están mezcladas en los nombres y son largos/incómodos para programar.
5. (Se verá más adelante) hay filas con frecuencia 0, filas duplicadas y un bloque fuera de orden temporal.


</details>

---
## 4. Preprocesamiento paso a paso

### 4.1 Nombres de columnas

⚠️ `¦Ì` es el símbolo **µ** guardado con una codificación de caracteres (asiática) y leído con otra (europea occidental). Pasa con frecuencia al exportar desde equipos o software extranjeros.

💡 Convención para todo el curso: **nombre corto + unidad**: `TOA_us`, `FREQ_MHz`, `PW_us`, `AMP_dBm`, `AOA_deg`, `PRI_us`.

Para renombrar se usa un **diccionario** `{nombre_viejo: nombre_nuevo}` (¡el Lab 0 sirve!).

In [ ]:
print(list(crudo.columns))     # copiamos los nombres exactamente como vienen

In [ ]:
nombres = {
    "TOA(¦Ìs)": "TOA_us",
    "Carrier Frequency(MHz)": "FREQ_MHz",
    "PW(¦Ìs)": "PW_us",
    "Amplitude(dBm)": "AMP_dBm",
    "AOA(deg)": "AOA_deg",
    "PRI(¦Ìs)": "PRI_us",
}
df = crudo.rename(columns=nombres)
df.columns

### 4.2 Eliminar columnas sin información

`df.drop(columns=[...])` elimina columnas. Primero confirmemos que `Unnamed: 6` no aporta nada:

In [ ]:
df["Unnamed: 6"].value_counts(dropna=False)     # cuenta cada valor distinto (incluidos vacíos)

In [ ]:
df = df.drop(columns=["Unnamed: 6"])
df.head(3)

### 4.3 Celdas vacías (NaN)

📘 En pandas una celda vacía se representa como **`NaN`** (*Not a Number*). `df.isna().sum()` cuenta cuántas hay por columna.

In [ ]:
df.isna().sum()

In [ ]:
df[df["PW_us"].isna()]          # mostrar SÓLO las filas donde PW está vacío

📘 La expresión `df[ condición ]` **filtra filas**: deja sólo las que cumplen la condición. Es la herramienta más importante de pandas.

Decisión del analista: un pulso sin PW está incompleto. Como son muy pocos, **los eliminamos** con `dropna`. *(Alternativa en otros casos: rellenar —imputar— con la mediana, documentándolo.)*

In [ ]:
n_antes = len(df)
df = df.dropna(subset=["PW_us"])
print(f"Eliminadas {n_antes - len(df)} filas con PW vacío · quedan {len(df):,}")

### 4.4 Registros "fantasma": ceros y valores centinela

📘 Algunos receptores escriben registros con **RF = 0, PW = 0 y una amplitud fija** (aquí −70 dBm) cuando se dispara el detector sin un pulso válido. A ese valor fijo se le llama **centinela**: no es una medición real.

Para buscar filas que cumplan **varias** condiciones se usan los operadores lógicos de pandas (cada condición **entre paréntesis**):

| Operador | Significado |
|---|---|
| `&` | Y (ambas) |
| `\|` | O (al menos una) |
| `~` | NO (invierte) |

In [ ]:
fantasma = (df["FREQ_MHz"] == 0) | (df["PW_us"] == 0)
print("Registros fantasma:", fantasma.sum())
df[fantasma].head()

⚠️ **Trampa frecuente:** "borrar todo lo que tenga amplitud ≤ −70 dBm". Mira cuántos pulsos **reales** (con RF y PW válidos) caen bajo −70 dBm:

In [ ]:
debiles_reales = (~fantasma) & (df["AMP_dBm"] <= -70)
print("Pulsos reales con AMP ≤ −70 dBm:", debiles_reales.sum())

🎯 Esos pulsos débiles son **lóbulos laterales** de la antena del radar o pulsos cerca del umbral del receptor: **información valiosa** para analizar el scan. *Regla de oro: nunca borres datos por un solo criterio sin mirar qué estás borrando.*

Eliminamos **sólo** los fantasma usando `~` (negación):

In [ ]:
n_antes = len(df)
df = df[~fantasma]
print(f"Eliminados {n_antes - len(df)} registros fantasma · quedan {len(df):,}")

### 4.5 Duplicados

📘 `df.duplicated()` marca las filas **idénticas** a una anterior (todas las columnas iguales). Suelen aparecer al concatenar archivos o por reenvío de datos del sensor.

In [ ]:
print("Filas exactamente duplicadas:", df.duplicated().sum())

In [ ]:
n_antes = len(df)
df = df.drop_duplicates()
print(f"Eliminadas {n_antes - len(df)} filas duplicadas · quedan {len(df):,}")

⚠️ **¿Por qué el orden de los pasos importa?** Un error común es eliminar duplicados **sólo por TOA** (`drop_duplicates(subset=["TOA_us"])`) *al principio*. Los registros fantasma comparten el TOA con un pulso real; si el fantasma aparece primero, **se conserva la basura y se borra el pulso bueno**. Comprobémoslo sobre los datos crudos:

In [ ]:
prueba = crudo.rename(columns=nombres).drop_duplicates(subset=["TOA_us"], keep="first")
print("Fantasmas que sobrevivirían :", (prueba["FREQ_MHz"] == 0).sum())
print("Filas eliminadas en total   :", len(crudo) - len(prueba))

💡 **Orden recomendado:** (1) nombres → (2) columnas inútiles → (3) vacíos → (4) fantasmas/centinelas → (5) duplicados exactos → (6) ordenar por TOA.

### 4.6 Orden temporal

📘 El análisis de PRI exige que los pulsos estén **ordenados por TOA**. `is_monotonic_increasing` verifica si una columna está en orden creciente.

In [ ]:
print("¿TOA ordenado?", df["TOA_us"].is_monotonic_increasing)

In [ ]:
df = df.sort_values("TOA_us").reset_index(drop=True)    # ordenar y renumerar filas 0, 1, 2, …
print("¿TOA ordenado?", df["TOA_us"].is_monotonic_increasing)
print("Pulsos válidos:", len(df))

💡 `reset_index(drop=True)` vuelve a numerar las filas desde 0. Sin esto, los números de fila quedan "salteados" por las filas eliminadas.

---
## 5. Seleccionar datos

| Quiero… | Código |
|---|---|
| Una columna | `df["FREQ_MHz"]` |
| Varias columnas | `df[["TOA_us", "FREQ_MHz"]]` |
| Filas por posición | `df.iloc[100:105]` |
| Filas que cumplen condición | `df[df["AMP_dBm"] > -45]` |
| Filas y columnas a la vez | `df.loc[df["AMP_dBm"] > -45, ["TOA_us", "AMP_dBm"]]` |

In [ ]:
df[["TOA_us", "FREQ_MHz", "PW_us"]].iloc[100:105]

In [ ]:
fuertes = df[df["AMP_dBm"] > -45]
print(f"Pulsos con AMP > −45 dBm: {len(fuertes)} de {len(df)} ({100*len(fuertes)/len(df):.1f} %)")

---
## 6. Parámetros derivados

### 6.1 Tiempo en segundos

El TOA viene en µs desde un origen arbitrario. Para graficar es más cómodo **segundos desde el primer pulso**:

In [ ]:
df["TOA_s"] = (df["TOA_us"] - df["TOA_us"].iloc[0]) / 1e6
print(f"Duración de la captura: {df['TOA_s'].iloc[-1]:.2f} s")

### 6.2 DTOA: diferencia de tiempo entre pulsos consecutivos

📘 $DTOA_n = TOA_n - TOA_{n-1}$ . En pandas: `df["TOA_us"].diff()` (el primer pulso queda `NaN` porque no tiene anterior).

🎯 **Si hay un solo emisor**, el DTOA es la **PRI**… *salvo* cuando se pierden pulsos (DTOA = 2·PRI, 3·PRI…) o cuando la antena deja de apuntarnos (huecos largos entre iluminaciones).

In [ ]:
df["DTOA_us"] = df["TOA_us"].diff()
df[["TOA_us", "TOA_s", "DTOA_us", "PRI_us"]].head(12)

### ✏️ Tu turno 2 · DTOA vs la columna PRI del equipo
Compara las columnas `DTOA_us` (calculada por ti) y `PRI_us` (calculada por el ESM). ¿Son iguales? ¿Cuándo difieren? Pista: mira las filas donde `PRI_us == 0`.

In [ ]:
# ✏️ escribe aquí: por ejemplo, filtra las filas donde PRI_us es 0 y mira su DTOA

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


```python
df[df["PRI_us"] == 0][["TOA_s", "DTOA_us", "PRI_us"]].head()
```
El ESM **reinicia la PRI a 0** cuando el hueco desde el pulso anterior es muy largo (> 20 ms en este equipo): interpreta que empieza una **nueva iluminación** de la antena. Nuestro `DTOA_us` en cambio muestra el hueco real (~5,6 s). Moraleja: **conoce cómo calcula tu equipo cada campo** antes de usarlo.


</details>

### 6.3 ¿Qué DTOA se repiten más?

`value_counts()` cuenta cuántas veces aparece cada valor. Como hay ruido de medición, primero **redondeamos** a la decena de µs.

In [ ]:
df["DTOA_us"].round(-1).value_counts().head(8)

🎯 Aparecen **dos** valores dominantes (≈ 900 y ≈ 1100 µs) y su suma (≈ 2000 µs). ¿Un emisor con dos PRI que alterna (*stagger*)? ¿Dos emisores distintos? Lo resolveremos con estadística (Lab 2) y análisis de PRI (Lab 3). **Anótalo como hipótesis.**

---
## 7. Primeros gráficos: el emisor "se muestra"

### 7.1 Amplitud vs tiempo

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df["TOA_s"], df["AMP_dBm"], ".", ms=3)
plt.xlabel("Tiempo desde el primer pulso (s)")
plt.ylabel("Amplitud (dBm)")
plt.title("Amplitud de cada pulso interceptado vs tiempo")
plt.grid(alpha=0.4)
plt.show()

🎯 **Interpretación.** Los pulsos llegan en **ráfagas periódicas**: es la antena del radar **girando**. Cada vez que el haz principal apunta al receptor, la amplitud sube al máximo; el resto del giro el haz apunta a otro lado y no detectamos nada (estamos bajo el umbral). El **tiempo entre ráfagas** es el **periodo de exploración** (*scan*).

### 7.2 Zoom en una iluminación
Filtremos un intervalo de tiempo (condición doble con `&`):

In [ ]:
t_ini, t_fin = 5.9, 6.4      # s  (ajusta si tu ráfaga cae en otro instante)
ventana = df[(df["TOA_s"] >= t_ini) & (df["TOA_s"] <= t_fin)]

plt.figure(figsize=(12, 4))
plt.plot(ventana["TOA_s"], ventana["AMP_dBm"], "o-", ms=3, lw=0.5)
plt.xlabel("Tiempo (s)"); plt.ylabel("Amplitud (dBm)")
plt.title(f"Una iluminación del haz ({len(ventana)} pulsos)")
plt.grid(alpha=0.4)
plt.show()

🎯 Se ve el **diagrama de radiación de la antena** del radar "dibujado" en el tiempo: un **lóbulo principal** (el máximo, ~ −36 dBm) y **lóbulos laterales** a ambos lados, 25–35 dB más abajo. Su forma y duración permiten estimar el **ancho de haz** (Lab 3).

### 7.3 Frecuencia, PW y DTOA vs tiempo
`plt.subplots(3, 1)` crea 3 gráficos apilados que comparten el eje de tiempo.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

ax[0].plot(df["TOA_s"], df["FREQ_MHz"], ".", ms=2)
ax[0].set_ylabel("RF (MHz)")
ax[0].set_title("Parámetros del PDW vs tiempo")

ax[1].plot(df["TOA_s"], df["PW_us"], ".", ms=2, color="tab:orange")
ax[1].set_ylabel("PW (µs)")

ax[2].plot(df["TOA_s"], df["DTOA_us"], ".", ms=2, color="tab:green")
ax[2].set_yscale("log")                     # escala logarítmica: ve a la vez 900 µs y 5 s
ax[2].set_ylabel("DTOA (µs, log)")
ax[2].set_xlabel("Tiempo (s)")

for a in ax:
    a.grid(alpha=0.4)
plt.tight_layout()
plt.show()

### ✏️ Tu turno 3 · Interpretar
Responde en esta celda (doble clic):

1. ¿La RF es **fija** o **ágil**? ¿Alrededor de qué valor? ¿Qué banda (IEEE / OTAN)?
2. ¿El PW es constante? ¿Alrededor de cuánto?
3. En el gráfico de DTOA (escala log) se ven "pisos" de valores. ¿Qué representa el piso ~10⁶–10⁷ µs?
4. ¿Cada cuántos segundos aparece una iluminación? → periodo de scan ≈ ___ s → ___ rpm.

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


1. RF **fija** ≈ 2950 MHz → banda **S** (IEEE) / **E** (OTAN). La dispersión de ±1–2 MHz es ruido de medición del receptor.
2. PW ≈ 1,2 µs, estable.
3. DTOA de ~5–6 s (≈ 5·10⁶ µs): **huecos entre iluminaciones** (la antena apunta a otro lado). Los pisos ~10³ µs son la PRI; ~2·10³, pulsos perdidos.
4. ≈ **6 s** entre iluminaciones → **10 rpm**: típico de un radar de **vigilancia aérea**.


</details>

### ✏️ Tu turno 4 · Practicar filtros
a) ¿Cuántos pulsos tienen DTOA entre 850 y 950 µs? ¿Y entre 1050 y 1150 µs?
b) Calcula la **mediana** de la RF de los pulsos con amplitud mayor a −45 dBm (haz principal). Pista: `.median()`.

In [ ]:
# ✏️ a)

# ✏️ b)

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


```python
# a)
n1 = df["DTOA_us"].between(850, 950).sum()
n2 = df["DTOA_us"].between(1050, 1150).sum()
print(n1, n2)          # cantidades muy parecidas → ambos valores se alternan
# b)
df[df["AMP_dBm"] > -45]["FREQ_MHz"].median()
```
`.between(a, b)` es un atajo para `(x >= a) & (x <= b)`.


</details>

---
## 8. Guardar el PDW limpio

Guardar el resultado evita repetir la limpieza y deja **trazabilidad**. Lo guardamos en Excel y en CSV (más liviano y universal).

In [ ]:
os.makedirs("resultados", exist_ok=True)
df.to_excel("resultados/D1_PDW_limpio.xlsx", index=False)
df.to_csv("resultados/D1_PDW_limpio.csv", index=False)
print("Guardado en la carpeta 'resultados/' ✔")

💡 En **Colab**, para descargar a tu computador: `from google.colab import files; files.download("resultados/D1_PDW_limpio.xlsx")`.

### 8.1 Bitácora de preprocesamiento
🎯 En un informe ELINT se debe declarar **qué se hizo a los datos**. Generemos la bitácora automáticamente:

In [ ]:
n_crudo = len(crudo)
n_vacios = int(crudo["PW(¦Ìs)"].isna().sum())
n_fantasma = int(((crudo["Carrier Frequency(MHz)"] == 0) | (crudo["PW(¦Ìs)"] == 0)).sum())
n_validos = len(df)
n_duplicados = n_crudo - n_vacios - n_fantasma - n_validos

bitacora = pd.DataFrame({
    "Paso": ["Filas exportadas por el ESM", "− Celdas PW vacías", "− Registros fantasma (RF=0 / PW=0, AMP=−70)",
             "− Duplicados exactos", "= Pulsos válidos"],
    "Filas": [n_crudo, n_vacios, n_fantasma, n_duplicados, n_validos],
})
bitacora

---
## 9. Resumen

### ✅ Checklist de preprocesamiento (aplícalo a cualquier captura real)
- [ ] Revisar `shape`, `head`, `tail`, `info` del archivo crudo.
- [ ] Renombrar columnas a **nombre + unidad**; verificar unidades reales (¿µs o s? ¿MHz o GHz?).
- [ ] Eliminar columnas vacías o sin información.
- [ ] Tratar celdas vacías (`isna`), documentando la decisión.
- [ ] Eliminar registros fantasma / centinela **mirando antes lo que se borra**.
- [ ] Eliminar duplicados **exactos** (no sólo por TOA).
- [ ] Ordenar por TOA y `reset_index`.
- [ ] Calcular `TOA_s` y `DTOA_us`.
- [ ] Guardar el resultado y la bitácora.

### Tips de pandas
| Quiero… | Código |
|---|---|
| Leer Excel / CSV | `pd.read_excel(ruta)` · `pd.read_csv(ruta)` |
| Renombrar / borrar columnas | `df.rename(columns={...})` · `df.drop(columns=[...])` |
| Contar vacíos | `df.isna().sum()` |
| Filtrar | `df[(cond1) & (cond2)]` · `~cond` · `.between(a, b)` |
| Duplicados | `df.duplicated().sum()` · `df.drop_duplicates()` |
| Ordenar | `df.sort_values("TOA_us").reset_index(drop=True)` |
| Diferencias | `df["TOA_us"].diff()` |
| Contar valores | `df["col"].value_counts()` |
| Guardar | `df.to_excel(ruta, index=False)` |

### Preguntas de repaso
1. ¿Por qué no conviene filtrar por amplitud para "limpiar" una captura?
2. ¿Qué tres cosas puede representar un DTOA en una captura de un solo emisor?
3. ¿Qué hipótesis tienes hasta ahora sobre este emisor (banda, PRI, scan, función)?

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


1. Porque los pulsos débiles suelen ser **reales** (lóbulos laterales, bordes del haz) y contienen información del patrón de antena y del scan.
2. (a) La **PRI**; (b) un múltiplo de la PRI por **pulsos perdidos**; (c) un **hueco entre iluminaciones** del haz.
3. Banda S (≈ 2950 MHz), PW ≈ 1,2 µs, dos PRI alternadas (≈ 900/1100 µs → posible stagger), scan circular ≈ 6 s (10 rpm) → probable **radar de vigilancia aérea**.


</details>

---
### ✅ Fin del Laboratorio 1

**Antes de continuar**, guarda tu trabajo: *Archivo → Guardar una copia en Drive* (Colab) o *File → Save* (Jupyter).

➡️ **Siguiente:** **Laboratorio 2 · Estadística descriptiva para el analista EW** (media, mediana, moda, percentiles, intervalos de confianza, histogramas, KDE, boxplot y violín).

<span style="font-size:11px;color:#7a7a7a">Curso de Guerra Electrónica · Análisis de Señales de Radar · Material de instrucción con datos sintéticos.</span>